<a href="https://colab.research.google.com/github/sairahul1526/pitch-deck-outline/blob/main/notebooks/ocr_benchmark_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pitch-deck OCR benchmark

This notebook is a provider-neutral Colab runner for the private OCR sample. It keeps raw PDFs and gold annotations in your Google Drive and writes only hashed benchmark artifacts. Run the native baseline first; enable Docling or PaddleOCR only after the runtime is ready.

## 1. Runtime and data paths

Use a GPU runtime only for the optional VLM cells. Upload either the local `data/raw` directory or the generated `pitch-deck-outline-raw.tar` archive into this Drive folder before running the benchmark. Do not upload permission emails or private gold transcriptions to GitHub.

In [ ]:
%pip install -q pypdf
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/pitch-deck-outline-data')
RAW_ROOT = DATA_ROOT / 'raw'
RUN_ROOT = DATA_ROOT / 'runs' / 'ocr'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('raw root:', RAW_ROOT)
print('exists:', RAW_ROOT.exists())

In [ ]:
import hashlib
import tarfile

archive_path = DATA_ROOT / 'pitch-deck-outline-raw.tar'
awesome_root = RAW_ROOT / 'awesome-pitch-decks' / 'pdfs'
if archive_path.exists() and not awesome_root.exists():
    with tarfile.open(archive_path, 'r') as archive:
        archive.extractall(DATA_ROOT, filter='data')

def unique_valid_pdfs(root: Path) -> list[Path]:
    unique: dict[str, Path] = {}
    for path in sorted(root.glob('*.pdf')):
        with path.open('rb') as handle:
            if handle.read(5) != b'%PDF-':
                continue
            digest = hashlib.sha256()
            for chunk in iter(lambda: handle.read(1024 * 1024), b''):
                digest.update(chunk)
        unique.setdefault(digest.hexdigest(), path)
    return sorted(unique.values())

awesome_pdfs = unique_valid_pdfs(awesome_root)
pitch_deckz_root = RAW_ROOT / 'huggingface' / 'skyforclouds__pitch-deckz' / 'files'
pitch_deckz_pdfs = unique_valid_pdfs(pitch_deckz_root)
print('Awesome Pitch Decks (unique valid PDFs):', len(awesome_pdfs))
print('Pitch Deckz (unique valid PDFs):', len(pitch_deckz_pdfs))
print('all PDFs:', len(awesome_pdfs) + len(pitch_deckz_pdfs))
assert awesome_pdfs, 'Upload data/raw or pitch-deck-outline-raw.tar into the Drive folder first'

In [ ]:
# The previous cell performs PDF-header validation and content-hash de-duplication.
sample = awesome_pdfs[:5]
print('sample:', [path.name for path in sample])

## 2. Native PDF baseline

This is the speed baseline. It is intentionally conservative: pages with little extracted text should be routed to OCR rather than silently accepted.

In [ ]:
import json
import time

from pypdf import PdfReader


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def native_extract(path: Path) -> dict:
    started = time.perf_counter()
    reader = PdfReader(str(path))
    pages = []
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        pages.append({'page_number': page_number, 'text': text, 'characters': len(text)})
    return {
        'engine': 'native-pypdf',
        'engine_version': 'colab-runtime',
        'source_file': str(path),
        'input_sha256': sha256_file(path),
        'latency_ms': round((time.perf_counter() - started) * 1000, 2),
        'pages': pages,
    }

sample = awesome_pdfs[:5]
native_results = [native_extract(path) for path in sample]
print([(Path(item['source_file']).name, len(item['pages'])) for item in native_results])

In [ ]:
native_report = RUN_ROOT / 'native-smoke.json'
native_report.write_text(json.dumps(native_results, indent=2) + '\n')
print(native_report)

## 3. Docling adapter (recommended first OCR candidate)

Docling is the first layout-aware candidate. It may install additional model assets on first use; keep its cache on the runtime or a disposable Drive cache.

In [ ]:
%pip install -q docling
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

def docling_extract(path: Path) -> dict:
    started = time.perf_counter()
    result = converter.convert(str(path))
    markdown = result.document.export_to_markdown()
    return {
        'engine': 'docling',
        'engine_version': 'colab-installed',
        'source_file': str(path),
        'input_sha256': sha256_file(path),
        'latency_ms': round((time.perf_counter() - started) * 1000, 2),
        'text': markdown,
    }

docling_result = docling_extract(sample[0])
print(docling_result['source_file'], len(docling_result['text']))

## 4. Expanded 50-page benchmark

The native baseline is measured page by page for exactly 50 pages. Docling is then run across the same five representative decks so layout-aware OCR output and latency can be reviewed together. This is a routing benchmark, not a quality claim until private gold transcriptions are reviewed.

In [ ]:
benchmark_decks = sample
native_page_rows = []
for path in benchmark_decks:
    reader = PdfReader(str(path))
    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ''
        native_page_rows.append({
            'source_file': path.name,
            'page_number': page_number,
            'characters': len(text),
            'empty': not bool(text.strip()),
        })

native_50_pages = native_page_rows[:50]
native_50_report = {
    'evidence_scope': 'expanded_50_page_sample',
    'engine': 'native-pypdf',
    'decks_considered': [path.name for path in benchmark_decks],
    'pages_sampled': len(native_50_pages),
    'characters': sum(row['characters'] for row in native_50_pages),
    'empty_pages': sum(row['empty'] for row in native_50_pages),
    'page_rows': native_50_pages,
}
native_50_path = RUN_ROOT / 'native-50-page-report.json'
native_50_path.write_text(json.dumps(native_50_report, indent=2) + '\n')

docling_runs = []
for path in benchmark_decks:
    result = docling_result if path == sample[0] else docling_extract(path)
    docling_runs.append({
        'source_file': path.name,
        'input_sha256': result['input_sha256'],
        'characters': len(result['text']),
        'latency_ms': result['latency_ms'],
        'native_pages': len(PdfReader(str(path)).pages),
    })
docling_5_report = {
    'evidence_scope': 'expanded_50_page_sample',
    'engine': 'docling',
    'documents_sampled': len(docling_runs),
    'source_pages_covered': sum(row['native_pages'] for row in docling_runs),
    'runs': docling_runs,
}
docling_5_path = RUN_ROOT / 'docling-5-deck-report.json'
docling_5_path.write_text(json.dumps(docling_5_report, indent=2) + '\n')

combined_report = {
    'evidence_scope': 'expanded_50_page_sample',
    'native': native_50_report,
    'docling': docling_5_report,
    'notes': (
        'Native metrics are page-level for exactly 50 pages. '
        'Docling runs cover the same five decks; compare quality after private review.'
    ),
}
combined_path = RUN_ROOT / 'ocr-expanded-report.json'
combined_path.write_text(json.dumps(combined_report, indent=2) + '\n')
print('native:', native_50_path)
print('docling:', docling_5_path)
print('combined:', combined_path)
print(
    'native pages:', native_50_report['pages_sampled'],
    'characters:', native_50_report['characters'],
    'empty:', native_50_report['empty_pages'],
)
print(
    'docling decks:', docling_5_report['documents_sampled'],
    'source pages:', docling_5_report['source_pages_covered'],
)

## 5. Optional PaddleOCR-VL adapter

Run this only on a GPU runtime after the native and Docling smoke checks pass. The API can change between PaddleOCR releases, so record the installed version in the report before comparing results.

In [ ]:
# Uncomment on a GPU runtime:
# %pip install -q paddlepaddle paddleocr
# from paddleocr import PaddleOCRVL
# vl_pipeline = PaddleOCRVL()
# vl_result = vl_pipeline.predict(str(sample[0]))
# print(vl_result)

## 6. Export a private benchmark artifact

Gold transcriptions and annotations are intentionally not generated here. Review the 50-page sample privately, then map each reviewed page into `BenchmarkCase` records in the repository's OCR harness.

In [ ]:
report = {
    'evidence_scope': 'colab_smoke',
    'native_results': native_results,
    'docling_result': docling_result,
    'notes': 'Smoke run only; not a quality claim.',
}
report_path = RUN_ROOT / 'ocr-smoke-report.json'
report_path.write_text(json.dumps(report, indent=2) + '\n')
print(report_path)